# Lab 1-01：文字怎麼變成 token

**目標**：自己寫一個最小的 BPE 切詞器，再拿它跟真實模型的切詞器比一比。

**你會學到**
- 模型看到的不是字，是一串整數（token id）
- BPE 怎麼從「一個 byte 一個 token」慢慢合併出常見片段
- 為什麼中文比英文貴、為什麼模型數不清字母

**前置**：只需要 Python。最後一段會下載 tiktoken 的詞表（約 3MB）。

**對應投影片**：「文字切成 token」「詞表與切分粒度」

## Step 1：電腦只認得數字

文字存進電腦時，本來就是一串 byte（0～255 的整數）。UTF-8 裡，一個英文字母是 1 個 byte，一個中文字是 3 個 byte。

In [1]:
text = "魯夫想成為海賊王"
raw = list(text.encode("utf-8"))
print(raw)
print(f"{len(text)} 個字 → {len(raw)} 個 byte")

[233, 173, 175, 229, 164, 171, 230, 131, 179, 230, 136, 144, 231, 130, 186, 230, 181, 183, 232, 179, 138, 231, 142, 139]
8 個字 → 24 個 byte


In [2]:
print(list("Luffy".encode("utf-8")))

[76, 117, 102, 102, 121]


## Step 2：最笨的切法——一個 byte 一個 token

直接把 byte 當 token，詞表只要 256 個，任何文字都切得了。
問題是序列太長：一句 8 個中文字就變成 24 個 token。模型每多看一個 token 都要花算力。

BPE（Byte Pair Encoding）的想法很樸素：**哪兩個 token 最常黏在一起，就把它們合併成一個新 token**，重複幾百次。

In [3]:
from pathlib import Path

corpus = Path("data/one_piece.txt").read_text(encoding="utf-8")
ids = list(corpus.encode("utf-8"))
print(f"語料 {len(corpus)} 個字元，{len(ids)} 個 byte")
print(corpus[:60])

語料 523 個字元，977 個 byte
魯夫是草帽海賊團的船長，他的夢想是成為海賊王。
魯夫小時候吃下了橡膠果實，身體可以像橡膠一樣伸長。
索隆是草帽海賊團的劍


## Step 3：數一數哪兩個 token 最常相鄰

In [4]:
from collections import Counter


def count_pairs(ids: list[int]) -> Counter:
    return Counter(zip(ids, ids[1:]))


pairs = count_pairs(ids)
for pair, n in pairs.most_common(5):
    print(pair, n, repr(bytes(pair).decode("utf-8", errors="replace")))

(101, 32) 17 'e '
(231, 154) 15 '�'
(154, 132) 15 '��'
(32, 116) 14 ' t'
(181, 183) 13 '��'


前幾名有英文的 `'e '`（e 後面接空白），也有印成 `�` 的組合：那是某個中文字的一部分。一個中文字要 3 個 byte，只拿到 2 個解不出字來。
下一步：把最常見的組合合併成一個新 id（從 256 開始編號），再重新數一次。

In [5]:
def merge(ids: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


print(merge([5, 6, 6, 7, 6, 7], (6, 7), 99))

[5, 6, 99, 99]


## Step 4：重複合併，就是在「訓練」切詞器

In [6]:
def train_bpe(ids: list[int], num_merges: int) -> dict[tuple[int, int], int]:
    merges: dict[tuple[int, int], int] = {}
    for k in range(num_merges):
        pairs = count_pairs(ids)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        new_id = 256 + k
        ids = merge(ids, best, new_id)
        merges[best] = new_id
    return merges


merges = train_bpe(list(corpus.encode("utf-8")), num_merges=200)

# 每個 token id 對應的原始 byte
vocab = {i: bytes([i]) for i in range(256)}
for (a, b), new_id in merges.items():
    vocab[new_id] = vocab[a] + vocab[b]

# 看看學到了哪些片段（只印解得出完整字、而且至少兩個字元的）
learned = []
for i in range(256, 256 + len(merges)):
    try:
        piece = vocab[i].decode("utf-8")
    except UnicodeDecodeError:
        continue  # 半個中文字，解不出來
    if len(piece) >= 2 and piece not in learned:
        learned.append(piece)
print(learned[:40])

['e ', ' t', 'he ', '草帽', '。\n', ' the ', 'at', '海賊', '草帽海賊', '草帽海賊團', '是草帽海賊團', '是草帽海賊團的', 'or', '，他', ' o', ' of', ' of the ', ' H', 'ate', 'ates', 'an', '魯夫', '成為', '，他想', ' i', 'ra', 'raw', 'Pi', 'Pir', 'Pirates', 'Pirates.', 'wor', '，他想成為', ' is', ' is the ', ' of the S', ' of the St', ' of the Straw', ' of the Straw H', ' of the Straw Hat']


「草帽海賊團」「想成為」這種常出現的片段，被合併成一個 token 了。這就是詞表的來源：**統計出來的，不是人定的**。

## Step 5：用學到的規則切新句子

In [7]:
def encode(text: str) -> list[int]:
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        # 照「訓練時越早學到越優先」的順序合併
        pair = min(count_pairs(ids), key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids


def decode(ids: list[int]) -> str:
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")


def show(piece: bytes) -> str:
    """把一個 token 印出來；半個中文字就印成 byte 的十六進位。"""
    try:
        return piece.decode("utf-8")
    except UnicodeDecodeError:
        return "<" + piece.hex() + ">"


for s in ["魯夫想成為海賊王", "索隆想成為劍豪", "喬巴喜歡吃棉花糖"]:
    toks = encode(s)
    print(f"{s}：{len(s.encode())} byte → {len(toks)} token", [show(vocab[t]) for t in toks])
    assert decode(toks) == s

魯夫想成為海賊王：24 byte → 7 token ['魯夫', '想', '成為', '海賊', '<e7>', '<8e>', '<8b>']
索隆想成為劍豪：21 byte → 12 token ['<e7b4>', '<a2>', '<e9>', '<9a>', '<86>', '想', '成為', '<e5>', '<8a8d>', '<e8>', '<b1>', '<aa>']
喬巴喜歡吃棉花糖：24 byte → 24 token ['<e5>', '<96>', '<ac>', '<e5>', '<b7>', '<b4>', '<e5>', '<96>', '<9c>', '<e6>', '<ad>', '<a1>', '<e5>', '<90>', '<83>', '<e6>', '<a3>', '<89>', '<e8>', '<8a>', '<b1>', '<e7>', '<b3>', '<96>']


`<e78e8b>` 這種是沒被合併的 byte：「王」在語料裡只出現一次，所以還是 3 個 byte 各自一個 token。
語料裡常見的片段切得很短；整句都沒看過的「喬巴喜歡吃棉花糖」只能退回一個 byte 一個 token。
真實模型的詞表是用幾兆個 token 的語料訓練出來的，道理完全一樣。

## Step 6：跟真實的切詞器比一比

`tiktoken` 是 OpenAI 的切詞器，`o200k_base` 是 GPT-4o 系列用的詞表（約 20 萬個 token）。

In [8]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
for s in ["Luffy wants to become the King of the Pirates.", "魯夫想成為海賊王。"]:
    toks = enc.encode(s)
    print(f"{len(s):>3} 字元 → {len(toks):>2} token", [show(enc.decode_single_token_bytes(t)) for t in toks])

 46 字元 → 11 token ['L', 'uffy', ' wants', ' to', ' become', ' the', ' King', ' of', ' the', ' Pirates', '.']
  9 字元 → 11 token ['<e9ad>', '<af>', '夫', '想', '成', '為', '海', '<e8b3>', '<8a>', '王', '。']


## Step 7：三個實務後果

1. **計費與 context 上限都按 token 算**：同樣的意思，中文通常比英文多吃 token。
2. **模型看不到字母**：它看到的是 `straw` `berry` 這種片段，所以數字母容易錯。
3. **數字被切得很碎**：大數字會被切成好幾段，算術因此不穩。

In [9]:
for s in ["strawberry", "1234567890", "3.14159"]:
    print(s, "→", [enc.decode([t]) for t in enc.encode(s)])

strawberry → ['st', 'raw', 'berry']
1234567890 → ['123', '456', '789', '0']
3.14159 → ['3', '.', '141', '59']


## 試試看

1. 把 `num_merges` 改成 20 和 1000，看看 Step 5 的 token 數怎麼變。
2. 在 `data/one_piece.txt` 加幾句你自己的句子，重新訓練，看看新詞會不會被合併。
3. 用 `enc.encode` 算一算：你最常寫的一段 prompt 有幾個 token？

## 小結

- 模型的輸入是一串 token id；BPE 把常一起出現的 byte 合併成新 token。
- 詞表是統計出來的，所以常見詞便宜、罕見詞和中文比較貴。
- 下一本：拿到 token 之後，模型怎麼猜下一個 token。